In [1]:
from ipyparallel import Client
import numpy as np
from nipype.interfaces.fsl import MELODIC
import os.path as op
import os

In [3]:
rc = Client()

In [6]:
db_path = '/home/dnl/Dropbox_ASU/Decision_Neuroscience_Lab/'
home_dir =  '/home/dnl/habitization/'
subs = db_path + 'Habitization/subjects.txt'
subs = list(np.loadtxt(subs,str))
subs = ['HAB06','HAB07']

In [7]:
def apply_melodic(in_tuple):
    sub, exp, run = in_tuple
    
    sub_path = home_dir + 'analysis/' + exp + '/' + sub + '/preproc/run_' + run + '/'
    
    if os.path.exists(sub_path):
        scan = sub_path + 'smoothed_timeseries.nii.gz'
        mask = sub_path + 'functional_mask.nii.gz'
        out_dir = sub_path + 'melodic'
        
        if not os.path.exists(out_dir):
            os.mkdir(out_dir)
        
        if os.listdir(out_dir) == []: #directory empty
            melodic_setup = MELODIC(in_files = scan, no_bet = True, bg_threshold = 10, tr_sec = 2,
                                   mm_thresh = .5, var_norm = True, out_stats = True, report = True, mask = mask,
                                   out_dir = out_dir)
            melodic_setup.run()
    return

In [8]:
in_tuples = []
for sub in subs:
    for exp in ['hab']:
        for run in range(1,13):
            in_tuples.append((sub,exp,str(run)))

In [9]:
dview = rc[0:10]
dview.block = True

dview.push(dict(home_dir=home_dir,
                ))
dview.execute("import numpy as np")
dview.execute("import os.path as op")
with dview.sync_imports():
    import os
    import numpy
    from nipype.interfaces.fsl import MELODIC
    
dview.map_sync(apply_melodic,in_tuples)


importing os on engine(s)
importing numpy on engine(s)
importing MELODIC from nipype.interfaces.fsl on engine(s)


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [6]:
#check for completion
exp = 'hab'
for sub in subs:
    for run in range(1,13):
        sub_path = home_dir + 'analysis/' + exp + '/' + sub + '/preproc/run_' + str(run) + '/melodic/'
        out_f = sub_path + 'melodic_IC.nii.gz'
        if not os.path.exists(out_f):
            print sub,run

HAB06 1
HAB06 2
HAB06 3
HAB06 4
HAB06 5
HAB06 6
HAB06 7
HAB06 8
HAB06 9
HAB06 10
HAB06 11
HAB06 12
HAB07 1
HAB07 2
HAB07 3
HAB07 4
HAB07 5
HAB07 6
HAB07 7
HAB07 8
HAB07 9
HAB07 10
HAB07 11
HAB07 12
